[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/S09_03_recomendadores.ipynb)

# S09 · 03 — Recomendar experiencias a los huéspedes

**Tiempo estimado:** 1 h 30 min · **Unidad del temario:** 3.11 (sistemas de recomendación)

**Contexto TurisData.** Además del alojamiento, TurisData vende **25 experiencias** (excursiones,
talleres, deportes). Tenemos 2 200 valoraciones de 300 usuarios (de 1 a 5 estrellas). Cada usuario
solo ha valorado entre 4 y 11 experiencias: la mayoría de las casillas de la tabla están **vacías**.
Queremos decirle a cada cliente *"esto te puede gustar"*.

**Al terminar sabrás:**
1. Montar la matriz usuario × experiencia y separar entrenamiento y prueba.
2. Construir un recomendador por **popularidad** (la referencia que hay que batir).
3. Recomendar **por contenido** (categorías) y por **filtrado colaborativo** (gente parecida).
4. Entender la **factorización de matrices** como "gustos ocultos".
5. Evaluar con **RMSE** (error de la nota) y **precision@k** (¿acierta lo que recomienda?).

*No usamos ninguna librería especializada: todo con numpy, pandas y scikit-learn.*

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

valoraciones = pd.read_csv(dato("valoraciones_experiencias.csv"))
experiencias = pd.read_csv(dato("experiencias.csv"))
print(valoraciones.shape, experiencias.shape)
valoraciones.head()

## 1. La matriz usuario × experiencia y la separación de prueba

**Idea.** Para saber si un recomendador es bueno hay que **esconderle** algunas valoraciones y ver si
las adivina. Reservamos un 20 % al azar como **prueba** (`prueba`) y construimos la matriz solo con el
80 % restante (`entrenamiento`). Las casillas vacías son `NaN`.

**Ejemplo resuelto:**

In [ ]:
prueba = valoraciones.sample(frac=0.2, random_state=SEMILLA)
entrenamiento = valoraciones.drop(prueba.index)

# filas = usuarios 1..300, columnas = experiencias 1..25
R = entrenamiento.pivot(index="id_usuario", columns="id_experiencia", values="valoracion")
R = R.reindex(index=range(1, 301), columns=range(1, 26))
M = R.notna().values                       # True donde hay valoración (matriz de "visto")
print("Matriz:", R.shape, "· casillas con dato:", f"{M.mean():.0%}")

media_global = entrenamiento["valoracion"].mean()
print("Nota media global:", round(media_global, 2))

# posiciones (fila, columna) de las valoraciones de prueba, para evaluar
fila_p = prueba["id_usuario"].values - 1
col_p = prueba["id_experiencia"].values - 1
real_p = prueba["valoracion"].values

def evaluar_rmse(P):
    """RMSE de una matriz de predicciones P (300 x 25) sobre las valoraciones de prueba."""
    pred = np.clip(P[fila_p, col_p], 1, 5)
    return float(np.sqrt(np.mean((pred - real_p) ** 2)))

**RMSE** = raíz del error cuadrático medio: *"de media, ¿en cuántas estrellas nos equivocamos?"*.
Cuanto más bajo, mejor. Punto de partida: predecir siempre la nota media global.

In [ ]:
P_global = np.full((300, 25), media_global)
print("RMSE prediciendo siempre la media global:", round(evaluar_rmse(P_global), 3))

## 2. Recomendador por popularidad

**Idea.** Recomendar a todos lo mejor valorado. Es simple, no personaliza y **casi siempre es la
referencia que hay que batir**. También es la única opción para un usuario nuevo (*arranque en frío*).

### Ejercicio 1
Calcula en `media_item` la nota media de cada experiencia **solo con la matriz de entrenamiento `R`**
(una `Series` de 25 valores). Después crea `P_pop`, una matriz de 300 × 25 en la que **todas las filas**
son `media_item` (`np.tile`), y guarda su error en `rmse_pop` con `evaluar_rmse`.

In [ ]:
# TODO: La media por columnas de un DataFrame promedia cada experiencia; np.tile sirve para repetir un vector en varias filas
...

assert len(media_item) == 25 and P_pop.shape == (300, 25), "media_item debe tener 25 valores y P_pop tamaño 300 x 25"
assert 0.95 < rmse_pop < 1.1, "El RMSE de popularidad debería rondar 1,0; revisa que uses R (solo entrenamiento)"
print(f"Ejercicio 1 correcto ✔ — RMSE popularidad = {rmse_pop:.3f}")

In [ ]:
top = media_item.sort_values(ascending=False).head(5)
print(experiencias.set_index("id_experiencia").loc[top.index, "nombre"].to_frame().assign(nota=top.round(2)))

## 3. Métrica de ranking: precision@k

**Idea.** El usuario no ve notas, ve **una lista corta**. Medimos si acertamos con ella:
**precision@k** = de las `k` experiencias recomendadas, ¿qué fracción le gustó de verdad (≥ 4 estrellas
en la prueba)? Se recomienda solo entre las que el usuario **no ha valorado** en entrenamiento.
Se promedia sobre los usuarios que tienen alguna experiencia "que le gusta" en la prueba.

Limitación: si recomendamos algo que le habría gustado pero nunca lo valoró, cuenta como fallo.
Por eso los valores son bajos y hay que **compararlos entre métodos**, no leerlos en absoluto.

### Ejercicio 2
Completa la función `precision_k(P, k=3)`. Para cada usuario de `relevantes`: (1) toma su fila de `P`,
(2) pon `-inf` en las experiencias que ya valoró en entrenamiento (fila de `M`), (3) quédate con las
`k` de mayor puntuación (`np.argsort`; suma 1 para pasar de posición a `id_experiencia`) y (4) calcula
qué fracción de esas `k` están en sus experiencias relevantes. Devuelve la media de todos los usuarios.

In [ ]:
relevantes = prueba[prueba["valoracion"] >= 4]      # lo que "le gusta" a cada usuario
print("Usuarios evaluables:", relevantes["id_usuario"].nunique())

def precision_k(P, k=3):
    aciertos = []
    # TODO: Recorre relevantes.groupby("id_usuario"); copia la fila P[u-1]; usa M[u-1] para ocultar lo ya visto; compara con set(...)
    ...
    return float(np.mean(aciertos))

p_pop = precision_k(P_pop)
azar = precision_k(np.random.default_rng(SEMILLA).random((300, 25)))
assert 0.09 < p_pop < 0.16, "precision@3 de popularidad debería rondar 0,12; revisa que ocultes lo ya valorado"
assert azar < p_pop + 0.02, "Recomendar al azar no debería superar claramente a popularidad; revisa la función"
print(f"Ejercicio 2 correcto ✔ — precision@3 popularidad = {p_pop:.3f} · azar = {azar:.3f}")

## 4. Recomendador basado en contenido

**Idea.** Si a un usuario le gustan las experiencias de *mar*, recomendarle más de *mar*. Cada
experiencia se describe por sus **características** (aquí, la categoría) y el usuario por **cuánto le
gustan** esas características, calculado con sus propias valoraciones.
Ventaja: funciona con experiencias **nuevas** que nadie ha valorado. Límite: solo ve lo que las
características describen.

Trabajamos con "residuos": lo que el usuario valora **por encima o por debajo** de la nota media de esa
experiencia (así separamos lo que es *bueno para todos* de lo que es *bueno para él*).

**Ejemplo resuelto:** matrices de características `F` y de residuos `C`.

In [ ]:
F = pd.get_dummies(experiencias.set_index("id_experiencia")["categoria"]).reindex(range(1, 26)).values.astype(float)
print("F (experiencia x categoría):", F.shape, "· categorías:", experiencias["categoria"].nunique())

media_item_v = media_item.fillna(media_global).values      # nota media de cada experiencia
C = np.where(M, R.values - media_item_v[None, :], 0.0)     # residuo (0 donde no hay valoración)

### Ejercicio 3
Calcula `perfil_usuario` (300 × 7): para cada usuario y categoría, la **suma de sus residuos en esa
categoría dividida entre (número de valoraciones en esa categoría + 3)**. El "+3" es prudencia: con pocas
valoraciones no nos fiamos. Pista: `C @ F` suma residuos por categoría y `M @ F` cuenta valoraciones
(convierte `M` a `float`).

In [ ]:
# TODO: Numerador y denominador son dos productos de matrices con F; el "+3" va solo en el denominador
...

assert perfil_usuario.shape == (300, 7), "perfil_usuario debe ser 300 usuarios x 7 categorías"
assert np.abs(perfil_usuario).max() < 3, "Los valores deben ser residuos pequeños (entre -3 y 3); revisa la división"
assert np.allclose(perfil_usuario[M.sum(1) == 0], 0), "Sin valoraciones el perfil debe ser 0"
print("Ejercicio 3 correcto ✔")

In [ ]:
# Predicción por contenido: nota media de la experiencia + afinidad del usuario con su categoría
P_cont = media_item_v[None, :] + perfil_usuario @ F.T
print("RMSE contenido:", round(evaluar_rmse(P_cont), 3), "· precision@3:", round(precision_k(P_cont), 3))

## 5. Filtrado colaborativo: "la gente como tú también valoró…"

**Idea.** No hace falta describir las experiencias. Miramos **quién valoró qué**:
- **Entre experiencias (item-item):** dos experiencias son *parecidas* si los mismos usuarios las puntúan
  por encima o por debajo de lo normal. Para predecir la nota de un usuario a una experiencia, usamos las
  notas que dio a las experiencias **parecidas**.
- **Entre usuarios (user-user):** dos usuarios son parecidos si valoran igual las mismas experiencias.

La "similitud" es el **coseno** entre dos columnas (o filas) de residuos: 1 = van a la par, 0 = nada
que ver, -1 = opuestos.

### Ejercicio 4
Calcula `S`, la matriz 25 × 25 de similitud coseno entre experiencias a partir de las **columnas de `C`**
(`cosine_similarity(C.T)` ya está importado). Pon la diagonal a 0 (una experiencia no se ayuda a sí misma)
con `np.fill_diagonal`. Después guarda en `similares_teide` los **nombres** de las 3 experiencias más
parecidas al *Ascenso al Teide* (id 1, columna 0).

In [ ]:
# TODO: Las experiencias son columnas de C. Ordena la fila 0 de S de mayor a menor similitud; recuerda que id = posición + 1
...

assert S.shape == (25, 25) and np.allclose(S, S.T), "S debe ser 25 x 25 y simétrica"
assert np.allclose(np.diag(S), 0), "La diagonal de S debe ser 0"
assert len(similares_teide) == 3 and all(isinstance(n, str) for n in similares_teide), "similares_teide: lista con 3 nombres"
print("Ejercicio 4 correcto ✔ — parecidas al Teide:", similares_teide)

In [ ]:
# Predicción item-item: nota media + media ponderada de los residuos del usuario en experiencias parecidas
P_item = media_item_v[None, :] + (C @ S.T) / (M.astype(float) @ np.abs(S).T + 1e-9)

# Predicción user-user: lo mismo mirando usuarios parecidos
Su = cosine_similarity(C)
np.fill_diagonal(Su, 0)
P_user = media_item_v[None, :] + (Su @ C) / (np.abs(Su) @ M.astype(float) + 1e-9)

for nombre, P in (("item-item", P_item), ("user-user", P_user)):
    print(f"{nombre}: RMSE = {evaluar_rmse(P):.3f} · precision@3 = {precision_k(P):.3f}")

## 6. Factorización de matrices: gustos ocultos

**Idea.** Supón que cada usuario tiene unos pocos **gustos ocultos** (p. ej. "aventura", "tranquilidad",
"cultura") y cada experiencia tiene una **intensidad** en cada uno. La nota esperada se obtiene
multiplicando ambos:  *nota ≈ media + sesgo del usuario + sesgo de la experiencia + gustos·intensidades*.
El algoritmo (aquí, mínimos cuadrados alternos) aprende los gustos y las intensidades a partir de las
casillas conocidas y **rellena las vacías**. Es la base de los grandes recomendadores.

**Ejemplo resuelto** (léelo con calma; no hace falta escribirlo):

In [ ]:
def factorizar(R, k=3, lam=5.0, iteraciones=30, semilla=SEMILLA):
    """Factorización con sesgos por mínimos cuadrados alternos. Devuelve la matriz completa de predicciones."""
    rng = np.random.default_rng(semilla)
    Mv = R.notna().values
    Y = np.nan_to_num(R.values)
    n_u, n_i = Y.shape
    mu = np.nanmean(R.values)
    P = rng.normal(0, 0.1, (n_u, k))          # gustos de cada usuario
    Q = rng.normal(0, 0.1, (n_i, k))          # intensidades de cada experiencia
    b_u, b_i = np.zeros(n_u), np.zeros(n_i)
    for _ in range(iteraciones):
        res = (Y - mu - b_i[None, :] - P @ Q.T) * Mv
        b_u = res.sum(1) / (Mv.sum(1) + 5)
        res = (Y - mu - b_u[:, None] - P @ Q.T) * Mv
        b_i = res.sum(0) / (Mv.sum(0) + 5)
        T = Y - mu - b_u[:, None] - b_i[None, :]
        for u in range(n_u):
            Qu = Q[Mv[u]]
            P[u] = np.linalg.solve(Qu.T @ Qu + lam * np.eye(k), Qu.T @ T[u, Mv[u]])
        for i in range(n_i):
            Pi = P[Mv[:, i]]
            Q[i] = np.linalg.solve(Pi.T @ Pi + lam * np.eye(k), Pi.T @ T[Mv[:, i], i])
    return mu + b_u[:, None] + b_i[None, :] + P @ Q.T

for k in (2, 3, 4, 6):
    print(f"k = {k} gustos ocultos → RMSE = {evaluar_rmse(factorizar(R, k=k)):.3f}")

### Ejercicio 5
Guarda en `P_mf` la matriz de `factorizar(R, k=3)` y calcula `rmse_mf` y `p_mf` (precision@3).
Después, en `mejor_rmse`, el nombre del método con menor RMSE de entre los que hay en el diccionario
`metodos` (que ya está preparado).

In [ ]:
# TODO: Ya tienes la función factorizar y las funciones de evaluación de los apartados anteriores
...

metodos = {"Popularidad": P_pop, "Contenido": P_cont, "Item-item": P_item, "User-user": P_user, "Factorización": P_mf}
tabla = pd.DataFrame({
    "RMSE": {n: evaluar_rmse(P) for n, P in metodos.items()},
    "precision@3": {n: precision_k(P) for n, P in metodos.items()},
}).round(3)
mejor_rmse = tabla["RMSE"].idxmin()
assert P_mf.shape == (300, 25), "P_mf debe ser 300 x 25"
assert rmse_mf < rmse_pop, "La factorización debería mejorar a popularidad; revisa k=3"
assert abs(p_mf - precision_k(P_mf)) < 1e-9, "p_mf debe ser precision_k(P_mf)"
print("Ejercicio 5 correcto ✔")
tabla

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
tabla["RMSE"].plot.barh(ax=ax[0], color="tab:blue"); ax[0].set_title("Error de la nota (RMSE) — menos es mejor"); ax[0].set_xlabel("estrellas")
tabla["precision@3"].plot.barh(ax=ax[1], color="tab:green"); ax[1].set_title("precision@3 — más es mejor"); ax[1].set_xlabel("proporción de aciertos")
plt.tight_layout(); plt.show()

**Lectura.** Fíjate en que cambiar de métrica cambia la clasificación: predecir bien la nota
(RMSE) no es lo mismo que **ordenar bien** la lista (precision@k). Con solo ~115 usuarios evaluables,
las diferencias pequeñas en precision@3 son **ruido**: no vendas una mejora de 0,01 al cliente.

## Mini-reto integrador: el recomendador de TurisData

Escribe `recomendar(id_usuario, n=3, P=P_mf)` que devuelva un `DataFrame` con las `n` experiencias
**que el usuario aún no ha valorado** con mayor puntuación prevista, con las columnas `nombre`,
`categoria`, `isla` y `nota_prevista`. Prueba con los usuarios 1 y 2 y comprueba que las
recomendaciones son distintas.

In [ ]:
# TODO: Copia P[id_usuario-1], pon -inf donde M[id_usuario-1] sea True, np.argsort(-...) y busca en experiencias por id
...

r1, r2 = recomendar(1), recomendar(2)
assert len(r1) == 3 and set(["nombre", "categoria", "isla", "nota_prevista"]) <= set(r1.columns), "Faltan columnas en el resultado"
vistas_1 = set(np.where(M[0])[0] + 1)
assert not (set(r1.index) & vistas_1), "No recomiendes experiencias que el usuario ya valoró"
assert set(r1.index) != set(r2.index), "Los usuarios 1 y 2 deberían recibir recomendaciones distintas"
print("Usuario 1:"); display(r1)
print("Usuario 2:"); display(r2)

## Preguntas de reflexión
1. ¿Qué método usarías para un **huésped nuevo** sin valoraciones? ¿Y para una **experiencia nueva**?
2. El recomendador por contenido salió peor que popularidad en RMSE. ¿Por qué crees que las categorías
   no bastan para explicar los gustos? ¿Qué otras características añadirías?
3. Si TurisData solo recomienda lo más popular, ¿qué experiencias quedarán siempre invisibles? ¿Es justo
   para los proveedores locales pequeños? *(riesgo de sesgo de popularidad)*.
4. ¿Qué datos personales usa este recomendador? ¿Qué debería saber el cliente según el RGPD?

## Qué has aprendido
- Popularidad, contenido, colaborativo y factorización: qué aporta y qué exige cada uno.
- Separar entrenamiento y prueba también en recomendación.
- RMSE mide la nota; **precision@k** mide la lista que ve el cliente.
- El arranque en frío se resuelve con popularidad o contenido.